In [1]:
!pip install -q transformers accelerate qwen-vl-utils torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 20.9 MB/s eta 0:00:00


In [2]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

print("Loading Multimodal Model into GPU...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_multimodal_description(text_input, image_path, video_path=None):
    """
    Generates a concise 2-3 sentence complaint description using:
    - text_input: Citizen keywords (required)
    - image_path: Photo file path or URL (required)
    - video_path: Video file path or URL (optional)
    """
    content = []

    # 1. Add Mandatory Image
    content.append({"type": "image", "image": image_path})

    # 2. Add Optional Video if provided
    if video_path is not None:
        content.append({"type": "video", "video": video_path})

    # 3. Add Prompt with Strict Output Formatting Rules
    prompt_text = (
        f"Citizen Input: '{text_input}'\n\n"
        "Task: Combine the citizen's keywords and the visual evidence (photo/video) into a formal complaint report.\n"
        "Rules:\n"
        "1. Write EXACTLY 2 to 3 clear sentences.\n"
        "2. State what the citizen is reporting, what the visual evidence shows, and the safety/health impact.\n"
        "3. Do not write bullet points, headers, or intros. Output ONLY the formal description paragraph."
    )
    content.append({"type": "text", "text": prompt_text})

    messages = [
        {
            "role": "system",
            "content": "You are an AI assistant for municipal civic complaints. Convert short keywords and visual evidence into concise, clear 2-3 sentence formal descriptions."
        },
        {"role": "user", "content": content}
    ]

    # Process vision and text inputs
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate response
    generated_ids = model.generate(**inputs, max_new_tokens=120, temperature=0.2, do_sample=True)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    response = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )

    return response[0].strip()

print("Multimodal Generator Ready!")

Loading Multimodal Model into GPU...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Multimodal Generator Ready!


In [3]:
import os
from PIL import Image

# 1. Create a local test image using PIL (no web download required)
test_image = Image.new('RGB', (400, 300), color=(120, 120, 120))
test_image.save("local_test_photo.jpg")

# -------------------------------------------------------------
# Case 1: Photo ONLY (Video is None)
# -------------------------------------------------------------
input_text_1 = "big pothole main road bike fall near bus stop"

output_1 = generate_multimodal_description(
    text_input=input_text_1,
    image_path="local_test_photo.jpg",  # Uses local generated file
    video_path=None
)

print("--- TEST CASE 1 (PHOTO ONLY) ---")
print(f"INPUT:  '{input_text_1}'")
print(f"OUTPUT: {output_1}")
print("=" * 60)

# -------------------------------------------------------------
# Case 2: Photo + Video (When video file is uploaded to Colab)
# -------------------------------------------------------------
# To test with a real video:
# 1. Click the 'Files' icon (folder) on the left sidebar in Colab.
# 2. Upload your video file (e.g. 'my_video.mp4').
# 3. Update video_path below and uncomment the code block:

"""
input_text_2 = "garbage stink road smell 3 days near school gate"

output_2 = generate_multimodal_description(
    text_input=input_text_2,
    image_path="local_test_photo.jpg",
    video_path="my_video.mp4"
)

print("--- TEST CASE 2 (PHOTO + VIDEO) ---")
print(f"INPUT:  '{input_text_2}'")
print(f"OUTPUT: {output_2}")
print("=" * 60)
"""

--- TEST CASE 1 (PHOTO ONLY) ---
INPUT:  'big pothole main road bike fall near bus stop'
OUTPUT: The citizen reports a significant pothole on the main road that has caused a bike rider to fall near the bus stop. The visual evidence shows a large, deep pothole in the center of the road, with no visible repair or warning signs, posing a serious risk to pedestrians and cyclists using the area. This condition poses a significant safety hazard, as it increases the likelihood of accidents, particularly among vulnerable road users such as cyclists and pedestrians.


'\ninput_text_2 = "garbage stink road smell 3 days near school gate"\n\noutput_2 = generate_multimodal_description(\n    text_input=input_text_2,\n    image_path="local_test_photo.jpg",\n    video_path="my_video.mp4"\n)\n\nprint("--- TEST CASE 2 (PHOTO + VIDEO) ---")\nprint(f"INPUT:  \'{input_text_2}\'")\nprint(f"OUTPUT: {output_2}")\nprint("=" * 60)\n'

In [4]:
import os
import torch
import cv2
import numpy as np
from PIL import Image

# 1. Helper to generate a 1-second dummy MP4 video if you don't have one uploaded yet
def create_dummy_video(filename="my_video.mp4", duration_sec=1, fps=10):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(filename, fourcc, fps, (320, 240))
    for i in range(duration_sec * fps):
        # Create a changing gray frame
        frame = np.full((240, 320, 3), (i * 20) % 255, dtype=np.uint8)
        out.write(frame)
    out.release()
    return filename

# Ensure test media files exist locally
if not os.path.exists("local_test_photo.jpg"):
    Image.new('RGB', (400, 300), color=(120, 120, 120)).save("local_test_photo.jpg")

if not os.path.exists("my_video.mp4"):
    create_dummy_video("my_video.mp4")

# -------------------------------------------------------------
# Case 2 Execution: Photo + Video
# -------------------------------------------------------------
input_text_2 = "garbage stink road smell 3 days near school gate"

output_2 = generate_multimodal_description(
    text_input=input_text_2,
    image_path="local_test_photo.jpg",
    video_path="my_video.mp4"
)

print("--- TEST CASE 2 (PHOTO + VIDEO) ---")
print(f"INPUT:  '{input_text_2}'")
print(f"OUTPUT: {output_2}")
print("=" * 60)

qwen-vl-utils using torchcodec to read video.
[transformers] Qwen2VL video processing does not apply the per-frame pixel cap the reference implementation (qwen-vl-utils) applies, so some videos cost far more tokens than they would there. In v5.22 the capped behavior will become the default and `cap_pixels_per_frame` will be removed. Pass `cap_pixels_per_frame=True` to adopt the reference behavior now, or `False` to keep the current behavior and silence this warning.


--- TEST CASE 2 (PHOTO + VIDEO) ---
INPUT:  'garbage stink road smell 3 days near school gate'
OUTPUT: The citizen reports experiencing a persistent odor emanating from garbage on a nearby road, lasting for three days near the school gate. The visual evidence suggests the presence of waste materials that have accumulated, contributing to an unpleasant smell that has affected the surrounding area, particularly impacting the health and well-being of students and staff at the school.
